In [ ]:
#импорт библиотек
import torch
import torch.nn as nn
from torchvision import models
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image, ImageOps
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
#Инициализация классов и путь, по которму лежит модель
MODEL_PATH = '/kaggle/input/notebook49a7f5233e/best_arch_resnet_50.pth' 

CLASSES = [
    "Achaemenid architecture",
    "American craftsman style",
    "American Foursquare architecture",
    "Ancient Egyptian architecture",
    "Art Deco architecture",
    "Art Nouveau architecture",
    "Baroque architecture",
    "Bauhaus architecture",
    "Beaux-Arts architecture",
    "Byzantine architecture",
    "Chicago school architecture",
    "Colonial architecture",
    "Deconstructivism",
    "Edwardian architecture",
    "Georgian architecture",
    "Gothic architecture",
    "Greek Revival architecture",
    "International style",
    "Novelty architecture",
    "Palladian architecture",
    "Postmodern architecture",
    "Queen Anne architecture",
    "Romanesque architecture",
    "Russian Revival architecture",
    "Tudor Revival architecture"
]
CLASSES = sorted(CLASSES)

In [ ]:
#Загрузка модели
def get_model(num_classes):
    model = models.resnet50(weights=None)

    in_features = model.fc.in_features

    model.fc = nn.Sequential(
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(256, num_classes)
    )

    return model

model = get_model(len(CLASSES))
model.load_state_dict(torch.load(MODEL_PATH, map_location='cuda'))
model.to('cuda')
model.eval()

print("Model is ready to use")

In [ ]:
#Подготовка картиинки и логика для предсказания
transform = A.Compose([
    A.Resize(height=448, width=448), # Тот же размер, что при обучении
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

def predict_image(image_path, model, transform, classes):
    image = Image.open(image_path).convert("RGB")

    image = ImageOps.exif_transpose(image) 
    
    plt.imshow(image)
    plt.axis('off')
    
    image_np = np.array(image)
    
    augmented = transform(image=image_np)
    image_tensor = augmented['image'].unsqueeze(0)
    image_tensor = image_tensor.to('cuda')
    
    with torch.no_grad():
        outputs = model(image_tensor)
        
        probs = torch.nn.functional.softmax(outputs, dim=1)[0]
        
        _, predicted_idx = torch.max(outputs, 1)
        predicted_class = classes[predicted_idx.item()]
        confidence = probs[predicted_idx.item()].item()
        
    print(f"Prediction: {predicted_class} ({confidence*100:.2f}%)")
    plt.title(f"{predicted_class} ({confidence*100:.1f}%)")
    plt.show()

In [ ]:
#Запуск предсказания
image_path = '/kaggle/input/test-1/1.jpeg'
predict_image(image_path, model, transform, CLASSES)